# Thanh: YOLO Detect person — chuẩn bị chạy ngày mai

Notebook và dataset phải được giữ **Private**. Hiện notebook chưa chạy train. Cập nhật 31/10/2026: lần chạy này dùng 477 ảnh train của `Thành.MOV`, 330 ảnh validation của `Thành (1).MOV`, cùng các nhãn người đã được duyệt. Chưa có tập test độc lập; kết quả là pilot của hai video này.

Người dùng yêu cầu 20 epoch, imgsz 1280 và hai GPU T4. Tìm checkpoint LoRA có thể tiếp tục trước; các vị trí đã kiểm tra chưa có bản được xác nhận, nên theo lựa chọn dự phòng sẽ dùng checkpoint LIGHT_CONTRAST đã lưu để fine-tune Detect thông thường một lớp. Các tensor không tương thích của đầu ra 8→1 lớp được khởi tạo lại; đây không phải tiếp tục adapter LoRA. Bản 8 lớp gốc được giữ riêng.

Trước khi chạy: bật Internet nếu môi trường chưa có Ultralytics 8.4.183, chọn **GPU T4 x2**, thêm dataset private `thanh-detect-person-and-pose13-reviewed-20261028` nếu chưa được gắn. Sau đó bấm **Save Version → Save & Run All** một lần. Không bấm chạy trong lúc chuẩn bị hôm nay.

In [ ]:
import os, sys, subprocess, importlib.metadata
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Cần chọn GPU T4 x2 trong Settings; không tự chuyển GPU.'
gpu_names = [torch.cuda.get_device_name(i) for i in range(2)]
assert all('T4' in x.upper() for x in gpu_names), f'Cần hai T4, hiện thấy: {gpu_names}'
print('GPU:', gpu_names)
try:
    installed = importlib.metadata.version('ultralytics')
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != '8.4.183':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.4.183'])
from ultralytics import settings
settings.update({'sync': False, 'wandb': False, 'comet': False, 'clearml': False, 'dvc': False, 'mlflow': False, 'raytune': False})
from ultralytics import YOLO, __version__
assert __version__ == '8.4.183', f'Ultralytics version unexpected: {__version__}'
print('Ultralytics:', __version__)

In [ ]:
from pathlib import Path
import hashlib, json, shutil, tempfile

INPUT = Path('/kaggle/input')
def unique_file(name):
    found = {p.resolve() for p in INPUT.rglob(name) if p.is_file()}
    if len(found) != 1:
        raise FileNotFoundError(f'Cần đúng một {name}; tìm thấy {len(found)}. Hãy gắn dataset Private đã chuẩn bị.')
    return next(iter(found))
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

bundle = unique_file('bundle_metadata.json').parent
metadata = json.loads((bundle / 'bundle_metadata.json').read_text(encoding='utf-8'))
checkpoint = bundle / 'weights' / 'LIGHT_CONTRAST_rsLoRA_best_E1.pt'
source_data = bundle / 'dataset'
manifest = json.loads(unique_file('frames_manifest.json').read_text(encoding='utf-8'))
report = json.loads((source_data / 'conversion_report.json').read_text(encoding='utf-8'))
assert metadata['checkpoint_sha256'] == sha256(checkpoint), 'Sai hash checkpoint'
# Input Private có thể là bundle cũ; kiểm split nguồn rồi chuẩn hóa bản sao làm việc bên dưới.
assert metadata['dataset_counts'] in ({'train': 330, 'val': 477}, {'train': 477, 'val': 330})
assert report['counts'] == metadata['dataset_counts']
assert report['boxes'] == ({'train': 1387, 'val': 1799} if report['counts']['train'] == 330 else {'train': 1799, 'val': 1387})
rows = {r['filename']: r for r in manifest['frames']}
assert len(rows) == 807 == len(manifest['frames'])
expected_videos = {'train': report['train_video_sha256'], 'val': report['val_video_sha256']}
seen = set()
for split, expected in metadata['dataset_counts'].items():
    images = sorted((source_data / 'images' / split).glob('*.jpg'))
    labels = sorted((source_data / 'labels' / split).glob('*.txt'))
    assert len(images) == len(labels) == expected
    assert {x.stem for x in images} == {x.stem for x in labels}
    boxes = 0
    for image in images:
        row = rows[image.name]
        assert row['source_video_sha256'] == expected_videos[split] and sha256(image) == row['image_sha256']
        assert image.name not in seen
        seen.add(image.name)
        lines = (source_data / 'labels' / split / (image.stem + '.txt')).read_text().splitlines()
        for line in lines:
            parts = line.split()
            assert len(parts) == 5 and parts[0] == '0'
            cx, cy, w, h = map(float, parts[1:])
            assert 0 <= cx - w / 2 <= cx + w / 2 <= 1 and 0 <= cy - h / 2 <= cy + h / 2 <= 1 and w > 0 and h > 0
        boxes += len(lines)
    assert boxes == report['boxes'][split]
    print(split, len(images), 'ảnh;', boxes, 'bbox person — hash ảnh khớp')
assert len(seen) == 807
work = Path(tempfile.mkdtemp(prefix='thanh_detect_'))
data_root = work / 'dataset'
shutil.copytree(source_data, data_root)

# Quyết định 31/10: train video nhiều ảnh hơn; không sửa dataset Kaggle chỉ đọc.
desired_train = '941205c3ad5f6d300b399be5102d6191227b0b9dd7397e76732252a39a5c19a4'
desired_val = 'b0a04470d591d46837fc81712cdbbfd74312444ae53e6f967131539a4b6770e5'
assert {report['train_video_sha256'], report['val_video_sha256']} == {desired_train, desired_val}
if report['train_video_sha256'] != desired_train:
    for kind in ('images', 'labels'):
        parent = data_root / kind
        (parent / 'train').rename(parent / '_split_swap')
        (parent / 'val').rename(parent / 'train')
        (parent / '_split_swap').rename(parent / 'val')
report.update(counts={'train': 477, 'val': 330}, boxes={'train': 1799, 'val': 1387},
              train_video_sha256=desired_train, val_video_sha256=desired_val)
(data_root / 'conversion_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
for part, expected in report['counts'].items():
    image_files = list((data_root / 'images' / part).glob('*.jpg'))
    label_files = list((data_root / 'labels' / part).glob('*.txt'))
    assert len(image_files) == len(label_files) == expected
    expected_hash = desired_train if part == 'train' else desired_val
    assert all(rows[p.name]['source_video_sha256'] == expected_hash for p in image_files)
    assert sum(len(p.read_text().splitlines()) for p in label_files) == report['boxes'][part]
print('Split sử dụng: train Thành.MOV 477 ảnh/1799 bbox; val Thành (1).MOV 330 ảnh/1387 bbox')
data_yaml = data_root / 'data.yaml'
out = Path('/kaggle/working/thanh_detect_person_20e_20261028')
out.mkdir(parents=True, exist_ok=False)
print('Checkpoint gốc đã xác nhận; kết quả mới sẽ ghi tại:', out)

In [ ]:
from ultralytics.nn.tasks import DetectionModel
base = YOLO(str(checkpoint))
assert base.names[0].lower() == 'person' and len(base.names) == 8
probe = DetectionModel(base.model.yaml, nc=1, ch=3, verbose=False)
src, dst = base.model.float().state_dict(), probe.state_dict()
matching = sum(k in src and src[k].shape == v.shape for k, v in dst.items())
assert (matching, len(dst)) == (696, 708), (matching, len(dst))
del probe, src, dst
print('Kiểm tra nạp:', matching, '/ 708 tensor tương thích; 12 tensor đầu ra khác kích thước.')

def metric_values(metrics):
    return {'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
            'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map)}
baseline = base.val(data=str(data_yaml), split='val', imgsz=1280, batch=8,
                    device=0, classes=[0], conf=0.001, iou=0.7, plots=True,
                    project=str(out), name='baseline', exist_ok=False)
(out / 'baseline_metrics.json').write_text(json.dumps(metric_values(baseline), indent=2))
print('Baseline person:', metric_values(baseline))

In [ ]:
# Train thường, không phải resume adapter LoRA. Model gốc trên Kaggle Input là chỉ đọc.
# Batch 16, optimizer auto và seed 0 theo giá trị mặc định của Ultralytics 8.4.183.
model = YOLO(str(checkpoint))
model.train(data=str(data_yaml), epochs=20, imgsz=1280, batch=16,
            device=[0, 1], seed=0, optimizer='auto', pretrained=True, resume=False,
            project=str(out), name='person20', exist_ok=False, plots=True)
best = Path(model.trainer.best)
assert best.is_file(), 'Không tìm thấy best.pt sau train'
assert sha256(checkpoint) == metadata['checkpoint_sha256'], 'Checkpoint gốc bị thay đổi'
print('Model 1 lớp mới:', best)

In [ ]:
trained = YOLO(str(best))
assert len(trained.names) == 1 and trained.names[0].lower() == 'person'
after = trained.val(data=str(data_yaml), split='val', imgsz=1280, batch=8,
                    device=0, classes=[0], conf=0.001, iou=0.7, plots=True,
                    project=str(out), name='after_training', exist_ok=False)
summary = {'baseline': metric_values(baseline), 'fine_tuned': metric_values(after),
           'validation_images': 330, 'train_images': 477, 'epochs_requested': 20,
           'imgsz': 1280, 'batch': 16, 'devices': gpu_names, 'classes': ['person'],
           'source_checkpoint_sha256': metadata['checkpoint_sha256'],
           'best_checkpoint_sha256': sha256(best), 'independent_test_available': False,
           'note': 'Pilot: one video train, one validation; no independent test.'}
(out / 'comparison.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print('Tải best.pt, comparison.json và các biểu đồ từ Output của phiên bản notebook Private.')